# Session 21: Measuring Model Performance

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Compute precision, recall and F1.
2. Read a classification report.
3. Draw a ROC curve and explain AUC.
4. Measure regression error with MAE and RMSE.
5. Explain R2 against a baseline.
6. Pick a metric that matches the business goal.

## Where We Are

In Session 20 the churn model scored 73 percent accuracy, yet it missed 93 of the 151 customers who actually left. One number hid the most important fact.

Today you learn the standard **metrics** for judging models. For classification: precision, recall, F1 and the ROC curve. For regression: MAE, RMSE and R2. Choosing the right metric is a business decision, so you will also practise matching metrics to goals.

## Setup

The setup cell rebuilds the churn model from Session 20, so `pred` and `proba` are ready.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             classification_report, roc_curve, roc_auc_score,
                             mean_absolute_error, root_mean_squared_error, r2_score)

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)
features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X_train, X_test, y_train, y_test = train_test_split(
    churn[features].astype(float), churn["left"],
    test_size=0.25, random_state=42, stratify=churn["left"])
scaler = StandardScaler().fit(X_train)
model = LogisticRegression().fit(scaler.transform(X_train), y_train)
pred = model.predict(scaler.transform(X_test))
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

## 1. Precision and Recall

In [ ]:
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall:   ", round(recall_score(y_test, pred), 3))
print("F1:       ", round(f1_score(y_test, pred), 3))

In [ ]:
print(classification_report(y_test, pred, target_names=["stayed", "left"]))

In [ ]:
for t in [0.3, 0.4, 0.5]:
    p = (proba >= t).astype(int)
    print(t, "precision", round(precision_score(y_test, p), 2),
          "recall", round(recall_score(y_test, p), 2))

## 2. ROC Curve and AUC

Each threshold gives a **true positive rate** (recall) and a **false positive rate** (the share of stayers flagged by mistake). The **ROC curve** draws all thresholds at once. The **AUC**, the area under the curve, is the chance that a random leaver gets a higher score than a random stayer: 0.5 is guessing, 1.0 is perfect.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, proba)
auc = roc_auc_score(y_test, proba)

plt.plot(fpr, tpr, label=f"Model, AUC {auc:.2f}")
plt.plot([0, 1], [0, 1], "--", label="Guessing")
plt.xlabel("False positive rate")
plt.ylabel("Recall")
plt.legend()
plt.show()

| Situation | Focus on | Why |
| --- | --- | --- |
| Classes are balanced | Accuracy | Every mistake costs about the same |
| Missing a case is costly | Recall | Disease screening, fraud, churn |
| A false alarm is costly | Precision | Spam filter, blocking a real payment |
| Need one balanced number | F1 | Comparing models on rare classes |
| Comparing probability models | ROC AUC | Independent of the threshold |

## 3. Regression Metrics

In [ ]:
flats = pd.read_csv(DATA_URL + "apartments.csv")
cols = ["area_m2", "rooms", "building_age", "distance_center_km", "elevator", "parking"]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    flats[cols].astype(float), flats["price_k"], test_size=0.25, random_state=42)

reg = LinearRegression().fit(Xa_train, ya_train)
price_pred = reg.predict(Xa_test)

In [ ]:
print("MAE: ", round(mean_absolute_error(ya_test, price_pred), 1))
print("RMSE:", round(root_mean_squared_error(ya_test, price_pred), 1))
print("R2:  ", round(r2_score(ya_test, price_pred), 3))

In [ ]:
baseline = np.full(len(ya_test), ya_train.mean())
print("Baseline MAE:", round(mean_absolute_error(ya_test, baseline), 1))
print("Baseline R2: ", round(r2_score(ya_test, baseline), 3))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Accuracy only | Judging a rare-class model by accuracy | Look at recall, precision and the confusion matrix. |
| Test on train | Computing metrics on the training data | Always report metrics on the test set. |
| Wrong input | Passing `pred` to `roc_auc_score` | Pass probabilities: `proba`, not 0 or 1 predictions. |
| No units | Reporting RMSE without units | Say 22 thousand, not just 22. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Metrics by Hand

From the confusion matrix `[[308, 41], [93, 58]]`, compute precision, recall and F1 with plain arithmetic, and check them against scikit-learn.

In [ ]:
# Write your code below

### In-Class Exercise 2: Best F1 Threshold

Try thresholds from 0.10 to 0.90 in steps of 0.05. Which one gives the highest F1 on the churn test set?

In [ ]:
# Write your code below

### In-Class Exercise 3: Which Metric?

For each case, choose the main metric and explain in one line: a cancer screening test; a filter that blocks suspicious bank payments; predicting delivery time in minutes; ranking customers for a call list.

In [ ]:
# Write your code below

### In-Class Exercise 4: Errors by Price

Compute the absolute error of each test apartment. Show the five flats with the largest errors and their real and predicted prices. What do they have in common?

In [ ]:
# Write your code below

## Summary

- Precision: of the flagged cases, how many are right.
- Recall: of the real cases, how many we found.
- F1 balances both; the threshold moves the balance.
- ROC AUC judges probabilities across all thresholds.
- MAE and RMSE measure regression error in the target's units.
- Always compare every metric with a simple baseline.

## What's Next

All the metrics today came from one random split. A different split would give different numbers. Next session you will use cross-validation for more reliable scores, and learn to spot models that memorise the training data.